# 10 - Build 1: a single-agent support triage system

Notebooks 01-09 took the SDK apart. This one puts it together on a real problem
and then tries to break it.

Everything is built here, in cells. By the end you will have a working agent that
reads a ticket, searches a knowledge base, looks up the account, applies a
written routing policy, drafts a customer reply, and files an escalation when
policy demands it — and a score telling you how often it got that right.

**What you will do**

1. Read the ticket queue and the policy it is judged against
2. Write four custom tools, and test them **with no API calls**
3. Assemble the system prompt and every option field
4. Run one ticket live and watch the loop
5. Score the result against the policy
6. Run the whole queue and get an accuracy number
7. Break it three ways on purpose

**Cost:** one ticket is roughly $0.02-0.06. The full queue is under $0.50.

> Run [notebook 01](01-Agent_SDK_Basics.ipynb) first if you have not verified your setup.

In [ ]:
# --- boot cell: run this first ------------------------------------------
import os, sys, json, re, asyncio
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Run notebook 01 first."
)

DATA = ROOT / "data"                  # tickets, accounts, KB, policy, expected answers
OUT  = ROOT / "out" / "single-agent"  # where this build writes its decisions
OUT.mkdir(parents=True, exist_ok=True)

MODEL = os.environ.get("COURSE_MODEL") or None

print("key loaded, model =", MODEL or "(SDK default)")
print("reading from :", DATA)
print("writing to   :", OUT)

## 10.1 The queue

Eight tickets. They are not eight variations of the same ticket — each one is
designed to punish a different shortcut.

In [ ]:
TICKETS = json.loads((DATA / "tickets.json").read_text(encoding="utf-8"))

print(f"{'id':<10} {'plan':<11} {'ch':<6} subject")
print("-" * 74)
for t in TICKETS:
    print(f"{t['id']:<10} {t['plan']:<11} {t['channel']:<6} {t['subject']}")

print(f"\n{len(TICKETS)} tickets\n")
print("TKT-1003 in full:\n")
t = next(t for t in TICKETS if t["id"] == "TKT-1003")
print(f"  from   : {t['customer']} ({t['plan']})")
print(f"  subject: {t['subject']}")
print(f"  body   : {t['body']}")

### What each ticket is testing

| Ticket | Looks like | The shortcut it punishes |
| --- | --- | --- |
| TKT-1001 | duplicate charge | escalating a refund policy lets tier 1 handle |
| TKT-1002 | "reset link never arrives" | answering "check your spam" without reading the account note |
| TKT-1003 | 429 errors | quoting the default rate limit to a customer with a contract |
| TKT-1004 | export question | raising priority because the plan is expensive |
| TKT-1005 | SSO blank screen | escalating instead of offering the documented 30-second fix |
| TKT-1006 | "is the api down??" | confirming an outage that is not happening |
| TKT-1007 | cancel + GDPR erasure | treating one ticket as one work item |
| TKT-1008 | wrong dashboard totals | letting plan or politeness set the severity |

The reference answers live in
[`data/expected.json`](../data/expected.json), each with the reasoning and the
trap written out.

In [ ]:
EXPECTED = {
    k: v for k, v in
    json.loads((DATA / "expected.json").read_text(encoding="utf-8")).items()
    if not k.startswith("_")
}

for tid in ("TKT-1004", "TKT-1008"):
    e = EXPECTED[tid]
    print(f"{tid}")
    print(f"  why  : {e['why']}")
    print(f"  trap : {e['trap']}")
    print(f"  want : {e['priority']} -> {e['queue']}, escalate={e['escalate']}\n")

Both are "a customer with a problem". One is P3 tier 1, the other is
P1 incident-response. **Impact sets priority; plan only breaks ties.** An agent
that has not internalised that gets one of them wrong every time.

## 10.2 The knowledge base and the policy

Two different kinds of document, and the difference matters.

The **knowledge base** is what the agent looks things up in — six articles it
searches at run time. The **routing policy** is what the agent *is* — it goes
into the system prompt verbatim, on every request.

In [ ]:
KB_DIR = DATA / "kb"

for path in sorted(KB_DIR.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    article = text.split("**Article ID:**")[1].split("\n")[0].strip()
    print(f"  {article}  {path.name:<28} {len(text):>5} chars")

POLICY = (DATA / "routing_policy.md").read_text(encoding="utf-8")
print(f"\n  KB-100  routing_policy.md{'':<12} {len(POLICY):>5} chars   <- goes into the prompt")

print("\n--- the hard stops ---")
print(POLICY.split("## Hard stops")[1].strip())

Those four rules are the ones a wrong answer actually costs something
for. Remember them — they come back in §10.7, and again in notebook 11 where a
hook enforces them in code.

## 10.3 Write the tools

Four tools on one **in-process MCP server** — ordinary async Python functions
that Claude can call. No separate process, no HTTP.

Each is four parts: a name, a description, an input schema, and a handler.

**The description is the API.** It is the only thing Claude reads when deciding
whether to call the tool, so it is written for Claude, not for you.

We start with a shared helper, because every tool returns the same shape.

In [ ]:
from claude_agent_sdk import tool, create_sdk_mcp_server, ToolAnnotations

def as_result(body: str, is_error: bool = False) -> dict[str, Any]:
    """Every tool result is {"content": [blocks]}. Build it in one place."""
    result: dict[str, Any] = {"content": [{"type": "text", "text": body}]}
    if is_error:
        # Claude reads this as "the call failed", not as odd-looking data.
        result["is_error"] = True
    return result


@tool(
    "search_kb",
    (
        "Search the support knowledge base and return the matching article "
        "sections in full. Pass 2-5 keywords describing the customer's problem, "
        "for example 'duplicate charge refund' or 'saml blank screen'. "
        "Always call this before drafting a reply - never answer a support "
        "question from memory."
    ),
    {"query": str},
    # readOnlyHint lets Claude batch this with other read-only calls.
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def search_kb(args: dict[str, Any]) -> dict[str, Any]:
    terms = [t for t in re.split(r"[^a-z0-9]+", str(args.get("query", "")).lower()) if len(t) > 2]
    if not terms:
        return as_result("Empty query. Pass keywords, e.g. 'refund duplicate charge'.", True)

    hits = []
    for path in sorted(KB_DIR.glob("*.md")) + [DATA / "routing_policy.md"]:
        content = path.read_text(encoding="utf-8")
        score = sum(content.lower().count(term) for term in terms)
        if score:
            hits.append((score, path.name, content))

    if not hits:
        return as_result(
            f"No knowledge base article matches {terms}. "
            "Do not guess an answer - say the KB does not cover it and escalate."
        )

    hits.sort(reverse=True)
    return as_result("\n\n".join(f"--- {n} (relevance {s}) ---\n{c}" for s, n, c in hits[:2]))


print("search_kb defined")

In [ ]:
@tool(
    "lookup_account",
    (
        "Look up the internal account record for a customer by their full name. "
        "Returns plan, monthly revenue, tenure, whether SSO is enforced, the "
        "contracted API rate limit, and an operational note. Call this whenever "
        "the reply depends on the customer's plan or entitlements."
    ),
    {"customer_name": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def lookup_account(args: dict[str, Any]) -> dict[str, Any]:
    accounts = json.loads((DATA / "accounts.json").read_text(encoding="utf-8"))
    name = str(args.get("customer_name", "")).strip()

    record = accounts.get(name) or next(
        (v for k, v in accounts.items() if k.lower() == name.lower()), None
    )
    if record is None:
        return as_result(f"No account found for '{name}'. Known: {', '.join(accounts)}", True)

    return as_result(json.dumps(record, indent=2))


print("lookup_account defined")

### Two schema styles

`search_kb` used the **short form** — a dict of name to Python type, where every
key is required:

```python
{"query": str}
```

The next two need enums, so they use **full JSON Schema**. The short form cannot
express an enum, a range, an array, or an optional field.

In [ ]:
QUEUES = ["tier1", "billing-ops", "api-platform", "identity-eng", "privacy", "incident-response"]


@tool(
    "file_escalation",
    (
        "File an escalation to a downstream team. Only call this when the "
        "routing policy says the ticket must leave tier 1. State the reason in "
        "one sentence, quoting the policy rule that applies."
    ),
    {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string", "description": "e.g. TKT-1003"},
            "queue": {"type": "string", "enum": QUEUES[1:]},   # tier1 is not an escalation
            "priority": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]},
            "reason": {"type": "string", "description": "One sentence, cite the policy rule"},
        },
        "required": ["ticket_id", "queue", "priority", "reason"],
    },
)
async def file_escalation(args: dict[str, Any]) -> dict[str, Any]:
    record = {"filed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), **args}
    with (OUT / "escalations.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record) + "\n")
    return as_result(
        f"Escalation filed to {args['queue']} at {args['priority']} for {args['ticket_id']}."
    )


print("file_escalation defined")

In [ ]:
@tool(
    "save_triage",
    (
        "Save the finished triage decision for one ticket. Call this exactly "
        "once per ticket, as the last step, after the knowledge base has been "
        "consulted. All seven fields are required by policy."
    ),
    {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string"},
            "category": {"type": "string",
                         "enum": ["billing", "login", "api", "sso", "data", "incident", "other"]},
            "priority": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]},
            "queue": {"type": "string", "enum": QUEUES},
            "kb_articles": {"type": "array", "items": {"type": "string"},
                            "description": "Article IDs actually used, e.g. ['KB-103']"},
            "draft_reply": {"type": "string", "description": "Customer-facing text"},
            "escalate": {"type": "boolean"},
            "escalation_reason": {"type": "string",
                                  "description": "Required when escalate is true"},
        },
        "required": ["ticket_id", "category", "priority", "queue",
                     "kb_articles", "draft_reply", "escalate"],
    },
)
async def save_triage(args: dict[str, Any]) -> dict[str, Any]:
    # Cheap policy checks, in code. Anything you can validate deterministically,
    # validate deterministically - do not ask the model to be careful instead.
    if args["escalate"] and not args.get("escalation_reason"):
        return as_result(
            "escalate is true but escalation_reason is missing. "
            "Call save_triage again with a one-line reason.", True)

    if not args["kb_articles"]:
        return as_result(
            "kb_articles is empty. Search the knowledge base and cite the article "
            "you relied on, or explain in escalation_reason why none applies.", True)

    record = {**args, "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    (OUT / f"{args['ticket_id']}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    return as_result(f"Triage saved: {args['priority']} -> {args['queue']}")


print("save_triage defined")

### Assemble the server

`create_sdk_mcp_server` bundles the tools. The key you later use in
`mcp_servers={"support": ...}` becomes the middle segment of every tool's fully
qualified name.

In [ ]:
support_server = create_sdk_mcp_server(
    name="support",
    version="1.0.0",
    tools=[search_kb, lookup_account, file_escalation, save_triage],
)

# Pattern: mcp__<mcp_servers key>__<tool name>. Two underscores between each part.
SUPPORT_TOOLS = [
    "mcp__support__search_kb",
    "mcp__support__lookup_account",
    "mcp__support__file_escalation",
    "mcp__support__save_triage",
]

print("server:", support_server["name"], "| type:", support_server["type"])
for name in SUPPORT_TOOLS:
    print("   ", name)

## 10.4 Test the tools with no API calls

A tool handler is a plain async function. Call it directly — no model, no cost,
no waiting. This is the fastest way to develop one, and it is worth doing
habitually: **get the tool right before you let an agent near it.**

In [ ]:
r = await search_kb.handler({"query": "duplicate charge refund"})
print("search_kb ->", r["content"][0]["text"][:240].replace("\n", " "), "...\n")

r = await lookup_account.handler({"customer_name": "Priya Nair"})
print("lookup_account ->\n", r["content"][0]["text"])

In [ ]:
# The failure paths matter more than the happy paths.

r = await lookup_account.handler({"customer_name": "Nobody At All"})
print("unknown customer -> is_error:", r.get("is_error"))
print("  ", r["content"][0]["text"][:110], "\n")

r = await save_triage.handler({
    "ticket_id": "TKT-DEMO", "category": "billing", "priority": "P3",
    "queue": "tier1", "kb_articles": [], "draft_reply": "hello", "escalate": False,
})
print("no KB article cited -> is_error:", r.get("is_error"))
print("  ", r["content"][0]["text"], "\n")

r = await save_triage.handler({
    "ticket_id": "TKT-DEMO", "category": "billing", "priority": "P1",
    "queue": "billing-ops", "kb_articles": ["KB-101"],
    "draft_reply": "hello", "escalate": True,      # escalate with no reason
})
print("escalate without a reason -> is_error:", r.get("is_error"))
print("  ", r["content"][0]["text"])

Three rules the policy states in prose are enforced here as `if`
statements, and each failure message tells Claude **what to do next** rather than
just reporting that something went wrong.

> **The habit worth taking from this notebook.** Anything you can check
> deterministically, check deterministically. The prompt says "cite the article
> you relied on"; `save_triage` is what makes it true. A prompt rule is a strong
> suggestion. A validated handler is a fact.

## 10.5 The system prompt

Two pieces: a short method section you write, and the routing policy pasted in
**verbatim from the file the support team owns**.

That split is the most valuable design decision in this build. The support team
edits Markdown, the agent's behaviour changes, and nobody touches Python. We
test that claim in §10.10.

In [ ]:
SYSTEM_PROMPT = f"""You are a support triage agent for a B2B SaaS company.

You handle one ticket per run. Your job is to categorise it, set a priority,
route it to a queue, draft a customer reply, and record the decision.

Method - follow it in order, every time:

1. Call search_kb with keywords from the ticket. Do this before anything else.
2. Call lookup_account for the customer if the answer depends on their plan,
   entitlements, or history.
3. Decide category, priority, and queue strictly from the policy below.
4. If the policy says the ticket leaves tier 1, call file_escalation.
5. Call save_triage exactly once, with all required fields.

Grounding rules:

- Every factual claim in your draft reply must come from a tool result. If you
  did not read it, do not write it.
- If the knowledge base does not cover the question, say so plainly in the reply
  and escalate. An honest escalation is a correct outcome, not a failure.
- Never invent an amount, a date, an incident ID, a root cause, or an SLA.

Keep your own commentary short. The saved record is the deliverable, not your
narration.

=== ROUTING POLICY (authoritative) ===
{POLICY}
=== END POLICY ==="""

split = SYSTEM_PROMPT.index("=== ROUTING POLICY")
print(f"system prompt: {len(SYSTEM_PROMPT):,} chars")
print(f"  method you wrote      : {split:,} chars")
print(f"  policy pasted verbatim: {len(SYSTEM_PROMPT) - split:,} chars "
      f"({(len(SYSTEM_PROMPT) - split) / len(SYSTEM_PROMPT):.0%} of the prompt)")

## 10.6 The options

Five settings decide whether this is safe to leave running.

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions

def build_options(max_turns: int = 20, budget_usd: float = 0.50) -> ClaudeAgentOptions:
    """The agent's whole configuration, in one place."""
    return ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        mcp_servers={"support": support_server},   # the key sets mcp__support__*
        tools=[],                    # AVAILABILITY: no built-ins exist at all
        allowed_tools=SUPPORT_TOOLS, # PERMISSION: these four run unprompted
        permission_mode="dontAsk",   # anything else -> denied, never a prompt
        max_turns=max_turns,         # bound the loop
        max_budget_usd=budget_usd,   # bound the bill
        setting_sources=[],          # no CLAUDE.md, no ambient config
        model=MODEL,
    )


o = build_options()
print("tools           ", o.tools, "  <- no Read, no Bash, no WebSearch. Nothing.")
print("allowed_tools   ", [n.split('__')[-1] for n in o.allowed_tools])
print("permission_mode ", o.permission_mode)
print("max_turns       ", o.max_turns, " | max_budget_usd", o.max_budget_usd)
print("setting_sources ", o.setting_sources)

Read it as five sentences:

> This is who you are. These are your tools. **Nothing else exists.** Nothing
> else runs. Here is when you stop.

`tools=[]` is the line people skip. Without it, `Read`, `Write`, `Bash`,
`WebSearch` and the rest sit in the agent's context — costing tokens every turn
and available for a cleverly worded ticket to reach for. With it, the agent's
entire surface is the four functions you just wrote.

## 10.7 The run loop

`query()` returns an async iterator of messages. Iterating it *is* running the
agent. Everything below just prints what goes past and records the outcome.

In [ ]:
from claude_agent_sdk import (
    query, AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

def preview(tool_input: dict, limit: int = 90) -> str:
    rendered = json.dumps(tool_input, default=str)
    return rendered if len(rendered) <= limit else rendered[: limit - 3] + "..."


async def triage_ticket(ticket: dict, verbose: bool = True, **opts) -> dict:
    """Run one ticket through the agent. Returns a small run summary."""
    prompt = (
        f"Triage this ticket.\n\n"
        f"ticket_id: {ticket['id']}\n"
        f"customer: {ticket['customer']}\n"
        f"plan on the ticket: {ticket['plan']}\n"
        f"channel: {ticket['channel']}\n"
        f"subject: {ticket['subject']}\n\n"
        f"body:\n{ticket['body']}"
    )

    summary = {"ticket_id": ticket["id"], "tool_calls": [], "subtype": None,
               "cost_usd": 0.0, "turns": 0, "denials": None}

    try:
        async for m in query(prompt=prompt, options=build_options(**opts)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        short = b.name.split("__")[-1]
                        summary["tool_calls"].append(short)
                        if verbose:
                            print(f"    -> {short}({preview(b.input)})")
                    elif isinstance(b, TextBlock) and verbose and b.text.strip():
                        print(f"       {b.text.strip()[:160]}")

            elif isinstance(m, ResultMessage):
                summary.update(subtype=m.subtype, cost_usd=m.total_cost_usd or 0.0,
                               turns=m.num_turns, denials=m.permission_denials)

    except Exception as exc:
        # A single-shot query() RAISES after yielding its error result, so the
        # ResultMessage branch above has usually already recorded the details.
        summary["error"] = str(exc)
        if verbose:
            print(f"    [!] run ended with an error: {exc}")

    return summary


print("triage_ticket ready")

## 10.8 Run one ticket

TKT-1003 — the rate-limit ticket. To get this right the agent has to notice that
the **applied** limit (300 rpm, in the account note) is below the **contracted**
limit (600 rpm), which KB-103 says makes it a platform fault rather than a
customer error.

Watch the tool calls. Their order tells you whether it followed the method.

In [ ]:
ticket = next(t for t in TICKETS if t["id"] == "TKT-1003")

print(f"{ticket['id']}  {ticket['subject']}")
summary = await triage_ticket(ticket)

print(f"\n  {summary['subtype']} | {summary['turns']} turns | "
      f"{len(summary['tool_calls'])} tool calls | ${summary['cost_usd']:.4f}")
print(f"  sequence: {' -> '.join(summary['tool_calls'])}")

In [ ]:
record = json.loads((OUT / "TKT-1003.json").read_text(encoding="utf-8"))

for field in ("category", "priority", "queue", "escalate", "kb_articles"):
    print(f"  {field:<14} {record.get(field)}")
if record.get("escalation_reason"):
    print(f"  {'reason':<14} {record['escalation_reason']}")

print("\n--- draft reply ---")
print(record["draft_reply"])

Check the reply against the four hard stops, and against what the tools
actually returned. Every factual claim should trace back to a tool result. If it
names a number, a date, or a cause that no tool produced, the agent invented it —
and that is the failure this whole design exists to prevent.

## 10.9 Score it

Judging by eye does not scale and does not survive a prompt change. Below is a
small scorer: it grades each field against `expected.json` and **independently**
re-checks the four hard stops in the drafted reply.

That independence matters. An evaluator that trusts the guardrail it is meant to
be evaluating is not an evaluator.

In [ ]:
# The four things routing_policy.md forbids absolutely.
HARD_STOPS = [
    (r"\b(has been|have been|was|were|is)\s+(permanently\s+)?deleted\b", "confirms deletion"),
    (r"\b(caused by|due to|because of)\s+(the\s+)?(2026[-.\d]*\s*)?(release|update|deploy|deployment)\b",
     "blames a release"),
    (r"\bpro[- ]?rated?\s+refund\b", "promises a prorated refund"),
    (r"\b(yes|confirm(ed)?),?\s+(that\s+)?(email|address|account)\s+is\s+registered\b",
     "confirms an email is registered"),
]

FIELDS = ("category", "priority", "queue", "escalate")


def score_ticket(ticket_id: str, record: dict) -> dict:
    e = EXPECTED[ticket_id]
    checks = {f: record.get(f) in e[f] for f in ("category", "priority", "queue")}
    checks["escalate"] = bool(record.get("escalate")) == bool(e["escalate"])
    checks["kb_articles"] = bool(set(record.get("kb_articles") or []) & set(e["kb_articles_any"]))

    reply = str(record.get("draft_reply", ""))
    violations = [why for pat, why in HARD_STOPS if re.search(pat, reply, re.IGNORECASE)]

    return {"ticket_id": ticket_id, "checks": checks,
            "passed": sum(checks.values()), "total": len(checks),
            "clean": not violations, "violations": violations,
            "got": {f: record.get(f) for f in FIELDS},
            "want": {f: e[f] for f in FIELDS},
            "why": e["why"], "trap": e["trap"]}


def load_run(out_dir) -> dict[str, dict]:
    return {p.stem: json.loads(p.read_text(encoding="utf-8"))
            for p in sorted(Path(out_dir).glob("TKT-*.json"))}


def print_scorecard(out_dir, label: str) -> dict:
    run = load_run(out_dir)
    if not run:
        print(f"\n  {label}: nothing in {out_dir} yet.")
        return {"tickets": 0, "accuracy": 0.0, "perfect": 0, "violations": 0}

    scores = [score_ticket(tid, rec) for tid, rec in run.items() if tid in EXPECTED]
    mark = lambda ok: f"{'ok' if ok else 'FAIL':^7}"          # noqa: E731
    cols = ("category", "priority", "queue", "escalate", "kb_articles")

    print(f"\n  {label}   ({len(scores)} tickets)")
    print("  " + "-" * 66)
    print(f"  {'ticket':<11}" + "".join(f"{h:^7}" for h in ("cat", "pri", "queue", "esc", "kb", "safe")))
    print("  " + "-" * 66)
    for s in scores:
        row = "".join(mark(s["checks"][c]) for c in cols)
        print(f"  {s['ticket_id']:<11}{row}{'ok' if s['clean'] else 'STOP':^7}")

    passed, total = sum(s["passed"] for s in scores), sum(s["total"] for s in scores)
    stops = sum(len(s["violations"]) for s in scores)
    print("  " + "-" * 66)
    print(f"  field accuracy: {passed / total:.0%}   "
          f"fully correct: {sum(1 for s in scores if s['passed'] == s['total'])}/{len(scores)}   "
          f"hard-stop violations: {stops}")

    for s in scores:
        if s["violations"]:
            print(f"\n  !! {s['ticket_id']} hard stop: {', '.join(s['violations'])}")
        elif s["passed"] < s["total"]:
            for f in FIELDS:
                if not s["checks"][f]:
                    print(f"     {s['ticket_id']} {f}: got {s['got'][f]!r}, "
                          f"expected one of {s['want'][f]}")

    return {"tickets": len(scores), "accuracy": passed / total,
            "perfect": sum(1 for s in scores if s["passed"] == s["total"]), "violations": stops}


print("scorer ready")

In [ ]:
s = score_ticket("TKT-1003", record)

print(f"  fields passed : {s['passed']}/{s['total']}")
for field, ok in s["checks"].items():
    print(f"    {'ok  ' if ok else 'FAIL'}  {field}")
print(f"  hard stops    : {'clean' if s['clean'] else s['violations']}")
print(f"\n  got  {s['got']}")
print(f"  want {s['want']}")
print(f"\n  why : {s['why']}")

## 10.10 The whole queue

Eight tickets, sequentially. This costs roughly **$0.20-0.50** and takes a few
minutes, so it is behind a flag — flip it when you are ready.

Notebook 11 reads these results for its head-to-head comparison, so run this
before you go there.

In [ ]:
RUN_FULL_QUEUE = False      # <-- set to True to spend real money

if not RUN_FULL_QUEUE:
    print("Skipped. Set RUN_FULL_QUEUE = True and re-run this cell.")
else:
    ledger = []
    for t in TICKETS:
        print(f"\n{t['id']}  {t['subject']}")
        s = await triage_ticket(t, verbose=False)
        ledger.append(s)
        print(f"   {s['subtype']} | {s['turns']} turns | "
              f"{len(s['tool_calls'])} tools | ${s['cost_usd']:.4f}")

    costs = sorted(e["cost_usd"] for e in ledger)
    print(f"\n  total   ${sum(costs):.4f}")
    print(f"  mean    ${sum(costs) / len(costs):.4f}")
    print(f"  min/max ${costs[0]:.4f} / ${costs[-1]:.4f}")

In [ ]:
totals = print_scorecard(OUT, "SINGLE AGENT")

Read the failures, not the percentage. Two questions are worth more
than the score:

- **Which field fails most?** If it is `priority`, the policy's ladder is
  ambiguous or under-explained. If it is `queue`, the queue descriptions are.
  Both are prompt problems, and both are fixable.
- **Is it wrong the same way twice?** A systematic error is a prompt bug. A
  scattered one is variance, and the fix is a stricter tool schema or a second
  pair of eyes — which is what notebook 11 is about.

## 10.11 Break it on purpose

Three experiments. Each removes one design decision and shows you what it was
holding up.

### Experiment 1 — take away a tool

`dontAsk` means unlisted tools are **denied**, not prompted. What does an agent
do when it cannot follow its own method?

In [ ]:
crippled = build_options()
crippled.allowed_tools = [n for n in SUPPORT_TOOLS if "search_kb" not in n]
print("allowed now:", [n.split("__")[-1] for n in crippled.allowed_tools], "\n")

t = next(t for t in TICKETS if t["id"] == "TKT-1005")
try:
    async for m in query(
        prompt=f"Triage this ticket.\n\nticket_id: {t['id']}\ncustomer: {t['customer']}\n"
               f"subject: {t['subject']}\n\nbody:\n{t['body']}",
        options=crippled,
    ):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock):
                    print(f"   TOOL {b.name.split('__')[-1]}")
                elif isinstance(b, TextBlock) and b.text.strip():
                    print(f"   ... {b.text.strip()[:200]}")
        elif isinstance(m, ResultMessage):
            print(f"\n   {m.subtype} | denials: {m.permission_denials}")
except Exception as exc:
    print("   raised:", str(exc)[:100])

Two behaviours are acceptable here and one is not.

**Acceptable:** it tries `search_kb`, is denied, and either works around the gap
honestly or reports that it cannot proceed. **Also acceptable:** it never tries,
and says so.

**Not acceptable:** it answers the SSO question from general knowledge. If it
does, the prompt's grounding rules are not strong enough — and that is exactly
the failure a QA reviewer catches in notebook 11.

### Experiment 2 — starve it of turns

`max_turns` counts tool-use round trips. The method needs at least three. Give
it two.

In [ ]:
t = next(t for t in TICKETS if t["id"] == "TKT-1001")

try:
    async for m in query(
        prompt=f"Triage this ticket.\n\nticket_id: {t['id']}\ncustomer: {t['customer']}\n"
               f"subject: {t['subject']}\n\nbody:\n{t['body']}",
        options=build_options(max_turns=2),
    ):
        if isinstance(m, ResultMessage):
            print(f"   subtype       : {m.subtype}")
            print(f"   turns         : {m.num_turns}")
            print(f"   cost          : ${m.total_cost_usd or 0:.4f}")
            print(f"   session_id    : {m.session_id}")
            print(f"   result present: {m.result is not None}")
except Exception as exc:
    print("   raised (expected):", str(exc)[:90])

print("\n   Nothing was saved - but the session_id survived, so this run is resumable")
print("   with a higher cap instead of starting over. See notebook 06, section 6.3.")

### Experiment 3 — change the policy, not the code

The claim in §10.5 was that the policy is content, not code. Test it: patch the
priority ladder so plan alone sets priority — the exact mistake the real policy
forbids — and re-run the export ticket that should be P3.

In [ ]:
original_prompt = SYSTEM_PROMPT

patched = original_prompt.replace(
    "Plan alone never sets priority",
    "Every Enterprise ticket is P1 regardless of impact. Plan alone never sets priority",
)
assert patched != original_prompt, "patch did not apply - has the policy text changed?"

SYSTEM_PROMPT = patched      # build_options() reads this global when called

t = next(t for t in TICKETS if t["id"] == "TKT-1004")     # a Pro export question
try:
    await triage_ticket(t, verbose=False)
    rec = json.loads((OUT / "TKT-1004.json").read_text(encoding="utf-8"))
    print(f"   with the patched policy    : {rec['priority']} -> {rec['queue']}")
    print(f"   the real policy expects    : {EXPECTED['TKT-1004']['priority']} -> "
          f"{EXPECTED['TKT-1004']['queue']}")
finally:
    SYSTEM_PROMPT = original_prompt
    print("\n   policy restored")

Behaviour changed with **no code change**. That is the property you
want: the people who own the policy can change what the agent does, and the
change is reviewable as a Markdown diff.

It is also the property that makes an eval harness non-optional. A prompt is
code, it regresses, and a Markdown diff that quietly moves three tickets from P3
to P1 looks completely harmless in review.

## 10.12 Where one agent runs out

Run the queue and look at the failures honestly. Four limits show up:

**1. One context does everything.** The same window holds the policy, the KB
articles it searched, the account record, and the draft. On a long ticket the
early instructions compete with the material it just read.

**2. Nothing checks the output.** The agent that wrote the reply is the agent
that judges it. `save_triage` catches missing fields; nothing catches a
confident, well-formatted, wrong claim.

**3. One prompt, four jobs.** Research, risk assessment, writing, and compliance
are different skills. A prompt good at all four is long, and long prompts blur.

**4. Everything runs sequentially.** Reading the KB and reading the account
record have no dependency on each other, yet one waits for the other.

Notebook 11 addresses all four — and costs three to five times as much doing it.
Whether that trade is worth making is a per-ticket decision, and you will have
the numbers to make it.

## 10.13 Exercises

1. **Add a tool.** `check_refund_eligibility(order_total, days_since_charge)`
   returning eligible/not with the reason from KB-101. Add it to the server and
   to `SUPPORT_TOOLS`. Does TKT-1001 get more reliable?
2. **Tighten a schema.** Make `save_triage` reject a `draft_reply` under 40
   characters or over 1,200. Watch the agent correct itself from the `is_error`
   message.
3. **Find the ambiguity.** Run the queue three times. Which field disagrees with
   itself most across runs? Rewrite that part of the policy and re-run.
4. **Cost.** Add `effort="low"` to `build_options()`. Re-run and score. How much
   cost went away, and did accuracy move at all?
5. **Grounding audit.** Take one draft reply and mark every factual claim. For
   each, name the tool result it came from. Any claim you cannot trace is a
   hallucination this design does not catch.

In [ ]:
# Scratch space for the exercises.

## Next

[**11 - Build 2: the multi-agent team**](11_build_multi_agent_triage.ipynb) —
the same problem with five agents, hooks that enforce the hard stops in code, and
a head-to-head against the results you just saved.

---

### References

- [Custom tools](https://code.claude.com/docs/en/agent-sdk/custom-tools)
- [Configure permissions](https://code.claude.com/docs/en/agent-sdk/permissions)
- [Modifying system prompts](https://code.claude.com/docs/en/agent-sdk/modifying-system-prompts)
- [How the agent loop works](https://code.claude.com/docs/en/agent-sdk/agent-loop)